# LLM-Proto: Build Your Own LLM From Scratch

This notebook validates the full pipeline end-to-end on a **tiny model** (~30M params).  
Designed to run in Google Colab (T4 GPU) or locally.

**Pipeline:**
1. Install dependencies
2. Train BPE tokenizer on sample data
3. Tokenize & save data as binary shards
4. Build model & inspect architecture
5. Run a mini training loop
6. Generate text samples
7. Visualize model internals

---

In [7]:
# ── Cell 1: Install Dependencies ──
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "torch>=2.1.0", "tokenizers", "datasets", "wandb", "safetensors",
    "matplotlib", "seaborn", "scikit-learn", "numpy", "pyyaml", "tqdm",
]:
    install(pkg)

print("All dependencies installed!")

All dependencies installed!


In [10]:
# ── Cell 2: Imports & Setup ──
import os, sys, torch
import numpy as np

# Add project root to path so we can import src/
if os.path.basename(os.getcwd()) != "LLM-Proto":
    # Colab: clone or mount
    if "google.colab" in sys.modules:
        # Option A: clone repo
        !git clone https://github.com/VlSePr/LLM-Proto
        os.chdir("LLM-Proto")
        

# Verify imports
from src.config import ModelConfig, TrainConfig, MODEL_CONFIGS, get_model_config
from src.model import TransformerLM
from src.tokenizer import LLMTokenizer
from src.utils import detect_environment, get_device, get_dtype, set_seed

env = detect_environment()
device = get_device()
dtype = get_dtype()
print(f"Environment: {env}")
print(f"Device: {device}")
print(f"Dtype: {dtype}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Environment: colab
Device: cuda
Dtype: torch.bfloat16
PyTorch: 2.10.0+cu128
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [11]:
# ── Cell 3: Inspect All Model Configs ──
print("Available model configurations:\n")
for name, cfg in MODEL_CONFIGS.items():
    est = cfg.param_count_estimate()
    print(f"  {name:8s} | dim={cfg.dim:5d} | layers={cfg.n_layers:2d} | "
          f"heads={cfg.n_heads:2d}/{cfg.n_kv_heads} | ffn={cfg.ffn_dim:5d} | "
          f"seq={cfg.max_seq_len:4d} | ~{est/1e6:.0f}M params")

Available model configurations:

  tiny     | dim=  512 | layers= 6 | heads= 8/4 | ffn= 1536 | seq=2048 | ~34M params
  small    | dim=  768 | layers=12 | heads=12/4 | ffn= 2048 | seq=2048 | ~93M params
  medium   | dim= 1024 | layers=24 | heads=16/4 | ffn= 2816 | seq=2048 | ~278M params
  base     | dim= 1280 | layers=24 | heads=20/4 | ffn= 3584 | seq=2048 | ~426M params
  large    | dim= 2048 | layers=32 | heads=32/8 | ffn= 5632 | seq=4096 | ~1374M params


In [12]:
# ── Cell 4: Build Tiny Model & Verify ──
set_seed(42)
model_config = get_model_config("tiny")
model = TransformerLM(model_config).to(device)
print(model.summary())

# Verify forward pass with random data
dummy_ids = torch.randint(0, model_config.vocab_size, (2, 128), device=device)
dummy_targets = torch.randint(0, model_config.vocab_size, (2, 128), device=device)

with torch.no_grad():
    out = model(dummy_ids, targets=dummy_targets)

print(f"\nForward pass OK!")
print(f"  Logits shape: {out['logits'].shape}")
print(f"  Loss: {out['loss'].item():.4f}")
print(f"  Expected initial loss (ln(vocab)): {np.log(model_config.vocab_size):.4f}")

TransformerLM Summary
  Layers: 6
  Hidden dim: 512
  Attention heads: 8 (KV heads: 4)
  Head dim: 64
  FFN dim: 1536
  Vocab size: 32000
  Max seq len: 2048
  Tie embeddings: True
  Total params: 35,265,024 (35.3M)
  Trainable params: 35,265,024 (35.3M)

Forward pass OK!
  Logits shape: torch.Size([2, 128, 32000])
  Loss: 10.4946
  Expected initial loss (ln(vocab)): 10.3735


In [13]:
# ── Cell 5: Test Generation (random weights — gibberish expected) ──
prompt_ids = torch.tensor([[1, 100, 200, 300]], dtype=torch.long, device=device)  # Dummy prompt
generated = model.generate(prompt_ids, max_new_tokens=20, temperature=0.8, top_k=50)
print(f"Generated token IDs: {generated[0].tolist()}")
print(f"(Gibberish expected — model is untrained)")

Generated token IDs: [1, 100, 200, 300, 13739, 21113, 23738, 20605, 23456, 21542, 20605, 8450, 15567, 14896, 20180, 2622, 20385, 16124, 31557, 31557, 31557, 17051, 9746, 10581]
(Gibberish expected — model is untrained)


In [ ]:
# ── Cell 6: Train BPE Tokenizer ──
# Trains a 32K BPE tokenizer on a small sample of FineWeb-Edu.
# This takes ~2-5 minutes depending on sample size.

from src.tokenizer import train_tokenizer_from_dataset

TOKENIZER_PATH = "tokenizer_data"
VOCAB_SIZE = 32_000
N_SAMPLES = 50_000  # Number of text samples to train tokenizer on (increase for production)

if os.path.exists(os.path.join(TOKENIZER_PATH, "tokenizer.json")):
    print(f"Tokenizer already exists at {TOKENIZER_PATH}, skipping training.")
    print("Delete the directory to retrain.")
else:
    print(f"Training BPE tokenizer (vocab={VOCAB_SIZE:,}, samples={N_SAMPLES:,})...")
    train_tokenizer_from_dataset(
        dataset_name="HuggingFaceFW/fineweb-edu",
        dataset_subset="sample-10BT",
        save_path=TOKENIZER_PATH,
        vocab_size=VOCAB_SIZE,
        num_samples=N_SAMPLES,
    )
    print("Tokenizer trained!")

# Test it
tok = LLMTokenizer(TOKENIZER_PATH)
test_text = "Hello, world! This is a test of the tokenizer."
ids = tok.encode(test_text, add_bos=True, add_eos=True)
decoded = tok.decode(ids)
print(f"\nTokenizer test:")
print(f"  Input:   {test_text}")
print(f"  Tokens:  {ids[:20]}{'...' if len(ids) > 20 else ''}")
print(f"  Decoded: {decoded}")
print(f"  Vocab size: {tok.vocab_size}")

Training BPE tokenizer (vocab=32,000, samples=50,000)...


TypeError: train_tokenizer_from_dataset() got an unexpected keyword argument 'output_dir'

In [ ]:
# ── Cell 7: Tokenize Data → Binary Shards ──
# Streams FineWeb-Edu, tokenizes, and saves as packed .bin files.
# For Colab prototyping, we limit to 5M tokens (~10 MB).

from src.data import tokenize_and_save

DATA_DIR = "data"
MAX_TOKENS = 5_000_000  # 5M tokens for quick prototyping (set None for full dataset)

if os.path.exists(os.path.join(DATA_DIR, "train_0000.bin")):
    print(f"Tokenized data already exists in {DATA_DIR}, skipping.")
    print("Delete the directory to re-tokenize.")
else:
    print(f"Tokenizing up to {MAX_TOKENS:,} tokens...")
    tokenize_and_save(
        dataset_name="HuggingFaceFW/fineweb-edu",
        dataset_subset="sample-10BT",
        tokenizer_path=TOKENIZER_PATH,
        output_dir=DATA_DIR,
        max_tokens=MAX_TOKENS,
    )
    print("Data tokenization complete!")

In [ ]:
# ── Cell 8: Verify DataLoader ──
from src.data import create_dataloader

train_loader = create_dataloader(DATA_DIR, model_config.max_seq_len, batch_size=4, split="train", num_workers=0)

batch = next(iter(train_loader))
input_ids, targets = batch

print(f"Batch input_ids shape: {input_ids.shape}")
print(f"Batch targets shape:   {targets.shape}")
print(f"Sample tokens: {input_ids[0, :20].tolist()}")
print(f"Decoded: {tok.decode(input_ids[0, :50].tolist())[:200]}")

In [ ]:
# ── Cell 9: Mini Training Loop (Quick Validation) ──
# Runs a small training loop (~100 steps) to verify everything works.
# NOT for production — just to confirm loss decreases.

import math, time
from contextlib import nullcontext
from src.utils import get_lr, set_seed

set_seed(42)

# Fresh tiny model
model_config = get_model_config("tiny")
model = TransformerLM(model_config).to(device)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=0.1,
                               betas=(0.9, 0.95), eps=1e-8)

# DataLoader
train_loader = create_dataloader(DATA_DIR, model_config.max_seq_len, batch_size=4,
                                  split="train", num_workers=0)
train_iter = iter(train_loader)

# Mixed precision
use_amp = dtype in (torch.float16, torch.bfloat16) and device.type == "cuda"
amp_ctx = torch.amp.autocast(device_type="cuda", dtype=dtype) if use_amp else nullcontext()
scaler = torch.amp.GradScaler(enabled=(use_amp and dtype == torch.float16))

# Training
N_STEPS = 100
model.train()
losses = []

print(f"Mini training: {N_STEPS} steps, batch=4, seq={model_config.max_seq_len}")
print(f"AMP: {use_amp}, dtype: {dtype}\n")

t0 = time.time()
for step in range(N_STEPS):
    try:
        input_ids, targets = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        input_ids, targets = next(train_iter)

    input_ids = input_ids.to(device)
    targets = targets.to(device)

    lr = get_lr(step, warmup_steps=10, max_steps=N_STEPS, peak_lr=6e-4, min_lr=6e-5)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    with amp_ctx:
        out = model(input_ids, targets=targets)
        loss = out["loss"]

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    losses.append(loss.item())
    if step % 10 == 0:
        print(f"  step {step:3d} | loss {loss.item():.4f} | ppl {math.exp(min(loss.item(), 20)):.1f} | lr {lr:.2e}")

elapsed = time.time() - t0
print(f"\nDone! {N_STEPS} steps in {elapsed:.1f}s")
print(f"Loss: {losses[0]:.4f} → {losses[-1]:.4f} (Δ = {losses[0] - losses[-1]:.4f})")
print(f"✓ Loss is decreasing — pipeline works!" if losses[-1] < losses[0] else "✗ Loss did not decrease — check for bugs")

In [ ]:
# ── Cell 10: Plot Training Loss ──
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Mini Training Loss Curve")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11: Generate Text After Mini Training ──
model.eval()

prompts = [
    "The meaning of life is",
    "Once upon a time",
    "In the year 2050",
]

for prompt in prompts:
    ids = tok.encode(prompt, add_bos=True)
    input_tensor = torch.tensor([ids], dtype=torch.long, device=device)
    output = model.generate(input_tensor, max_new_tokens=50, temperature=0.8, top_k=50)
    text = tok.decode(output[0].tolist())
    print(f"Prompt: {prompt}")
    print(f"Output: {text[:200]}")
    print("-" * 60)

In [ ]:
# ── Cell 12: Visualize Model Internals ──
from src.visualize import (
    plot_weight_distributions,
    plot_activation_stats,
    plot_token_loss_heatmap,
)

# Weight distributions
fig = plot_weight_distributions(model)
plt.show()

# Activation stats
sample_ids = next(iter(train_loader))[0][:1].to(device)
fig = plot_activation_stats(model, sample_ids)
plt.show()

# Per-token loss heatmap
fig = plot_token_loss_heatmap(model, sample_ids, tok, max_len=64)
plt.show()

In [ ]:
# ── Cell 13: Save & Load Checkpoint Test ──
from src.utils import save_checkpoint, load_checkpoint

CKPT_DIR = "checkpoints"
train_config = TrainConfig(use_wandb=False)

# Save
save_checkpoint(model, optimizer, step=99, loss=losses[-1],
                model_config=model_config, train_config=train_config,
                checkpoint_dir=CKPT_DIR)
print(f"Checkpoint saved to {CKPT_DIR}/")

# Load into a fresh model
model2 = TransformerLM(model_config).to(device)
ckpt = load_checkpoint(CKPT_DIR, "latest", model2, device=device)
print(f"Loaded checkpoint from step {ckpt['step']}")

# Verify weights match
for (n1, p1), (n2, p2) in zip(model.named_parameters(), model2.named_parameters()):
    assert torch.equal(p1, p2), f"Mismatch at {n1}"
print("✓ Checkpoint save/load verified — weights match!")